# Qwen2-0.5B LoRA — Baseline (Notebook 1 of 2)

Trains a single fixed-hyperparameter LoRA baseline on one serialization, logs full metrics
per epoch + final, saves the model for later XAI use, and writes a `shared_config.json`
handoff file so the companion Optuna notebook can reuse the *exact same* train/val split.

Set `SERIALIZATION` below, then **Save Version → Save & Run All** so the outputs
(under `/kaggle/working/`) become attachable as an input to the Optuna notebook.


In [ ]:
%pip install -q transformers datasets accelerate peft scikit-learn pandas matplotlib
%pip uninstall -y torchao -q


In [ ]:
import os, re, time, json, math, random, gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from datasets import Dataset

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, TrainerCallback
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ------------------------------------------------------------------
# CHOOSE SERIALIZATION HERE
# ------------------------------------------------------------------
SERIALIZATION = "prompt_style"   # "exact" | "human_readable" | "prompt_style" | "json"

DATA_DIR = "/kaggle/input/datasets/tamislam/llm-serialisation-dataset-final/kickstarter_serializations"
TRAIN_FILE = f"{DATA_DIR}/{SERIALIZATION}/kickstarter_llm_train.csv"
TEST_FILE  = f"{DATA_DIR}/{SERIALIZATION}/kickstarter_llm_test.csv"

MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
OUTPUT_DIR = "/kaggle/working"
BASELINE_MODEL_DIR = f"{OUTPUT_DIR}/baseline_model"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/baseline_checkpoints"

MAX_LENGTH = 512
MAX_NEW_TOKENS = 12
GENERATION_BATCH_SIZE = 32
VAL_FRACTION = 0.15

SYSTEM_PROMPT = (
    "Predict the final Kickstarter funding value as log1p(USD) "
    "from the provided campaign features. Output only the numeric value."
)

BASELINE_PARAMS = dict(
    r=16, lora_alpha=32, lora_dropout=0.05,
    learning_rate=1e-4, num_train_epochs=4,
    train_batch_size=8, gradient_accumulation_steps=4,
    weight_decay=0.01, warmup_ratio=0.03, lr_scheduler_type="cosine",
)

print("Serialization:", SERIALIZATION)


## Load data, carve out validation split (fixed IDs saved for handoff)

In [ ]:
train_full_df = pd.read_csv(TRAIN_FILE, keep_default_na=False)
test_df = pd.read_csv(TEST_FILE, keep_default_na=False)

required_columns = {"id", "text", "target", "target_usd"}
assert required_columns.issubset(train_full_df.columns)
assert required_columns.issubset(test_df.columns)

for col in ["target", "target_usd"]:
    train_full_df[col] = pd.to_numeric(train_full_df[col], errors="raise")
    test_df[col] = pd.to_numeric(test_df[col], errors="raise")

assert len(set(train_full_df["id"]).intersection(set(test_df["id"]))) == 0

val_size = int(len(train_full_df) * VAL_FRACTION)
shuffled = train_full_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = shuffled.iloc[:val_size].reset_index(drop=True)
train_df = shuffled.iloc[val_size:].reset_index(drop=True)

print("Train rows:", len(train_df), "| Val rows:", len(val_df), "| Test rows:", len(test_df))

fallback_log_target = float(train_full_df["target"].median())
max_reasonable_log_target = float(train_full_df["target"].max() + 2.0)
print("Fallback log target:", fallback_log_target, "| Max accepted:", max_reasonable_log_target)


## Tokenizer & shared helper functions

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_bf16_supported():
    TRAIN_DTYPE, USE_BF16, USE_FP16 = torch.bfloat16, True, False
else:
    TRAIN_DTYPE, USE_BF16, USE_FP16 = torch.float16, False, True
print("dtype:", TRAIN_DTYPE)


def build_training_example(example):
    target_text = f"{float(example['target']):.4f}"
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(example["text"])},
    ]
    full_messages = prompt_messages + [{"role": "assistant", "content": target_text}]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)["input_ids"]
    full = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
    input_ids = full["input_ids"]; attention_mask = full["attention_mask"]
    labels = input_ids.copy()
    prompt_length = min(len(prompt_ids), len(labels))
    labels[:prompt_length] = [-100] * prompt_length
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


class CausalLMDataCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        batch_input_ids, batch_attention_mask, batch_labels = [], [], []
        for item in features:
            pad_len = max_len - len(item["input_ids"])
            batch_input_ids.append(item["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            batch_attention_mask.append(item["attention_mask"] + [0] * pad_len)
            batch_labels.append(item["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }

data_collator = CausalLMDataCollator(tokenizer)
NUMBER_PATTERN = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

def parse_generated_log_target(text):
    match = NUMBER_PATTERN.search(str(text))
    if match is None: return np.nan
    try: value = float(match.group(0))
    except Exception: return np.nan
    if not np.isfinite(value) or value < 0 or value > max_reasonable_log_target: return np.nan
    return value

def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_regression_full(y_true_log, pred_log, actual_usd, pred_usd):
    """Full 9-metric suite: MAE/MSE/RMSE/R2 on log and USD scale, plus RMSLE."""
    return {
        "MAE_log": mean_absolute_error(y_true_log, pred_log),
        "MSE_log": mean_squared_error(y_true_log, pred_log),
        "RMSE_log": rmse(y_true_log, pred_log),
        "R2_log": r2_score(y_true_log, pred_log),
        "MAE_USD": mean_absolute_error(actual_usd, pred_usd),
        "MSE_USD": mean_squared_error(actual_usd, pred_usd),
        "RMSE_USD": rmse(actual_usd, pred_usd),
        "R2_USD": r2_score(actual_usd, pred_usd),
        "RMSLE": np.sqrt(mean_squared_log_error(actual_usd, pred_usd)),
    }

def build_inference_prompt(feature_text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": str(feature_text)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def generate_predictions(model, eval_df, batch_size=GENERATION_BATCH_SIZE):
    model.eval(); model.config.use_cache = True; tokenizer.padding_side = "left"
    device = next(model.parameters()).device
    prompts = [build_inference_prompt(x) for x in eval_df["text"]]
    generated_texts = []
    t0 = time.perf_counter()
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        prompt_width = inputs["input_ids"].shape[1]
        new_tokens = generated_ids[:, prompt_width:]
        generated_texts.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    gen_time = time.perf_counter() - t0
    pred_log_raw = np.array([parse_generated_log_target(t) for t in generated_texts], dtype=float)
    valid_mask = np.isfinite(pred_log_raw)
    pred_log = pred_log_raw.copy(); pred_log[~valid_mask] = fallback_log_target
    tokenizer.padding_side = "right"; model.config.use_cache = False
    return pred_log, valid_mask, generated_texts, gen_time


## Per-epoch metric logger (callback) + train/eval routine with checkpointing

In [ ]:
class EpochMetricsCallback(TrainerCallback):
    """At the end of every epoch: run generation-based eval on eval_df,
    compute the full 9-metric suite, append a row to <output_dir>/epoch_metrics.csv."""
    def __init__(self, eval_df, metrics_csv_path):
        self.eval_df = eval_df
        self.metrics_csv_path = metrics_csv_path
        self.rows = []

    def on_epoch_end(self, args, state, control, **kwargs):
        model = kwargs["model"]
        pred_log, valid_mask, _, gen_time = generate_predictions(model, self.eval_df)
        y_true_log = self.eval_df["target"].to_numpy(dtype=float)
        actual_usd = self.eval_df["target_usd"].to_numpy(dtype=float)
        pred_usd = np.clip(np.expm1(pred_log), a_min=0, a_max=None)
        metrics = evaluate_regression_full(y_true_log, pred_log, actual_usd, pred_usd)
        metrics["epoch"] = round(state.epoch, 4)
        metrics["parse_success_rate"] = float(valid_mask.mean())
        metrics["generation_time_seconds"] = gen_time
        self.rows.append(metrics)
        pd.DataFrame(self.rows).to_csv(self.metrics_csv_path, index=False)
        print(f"  [epoch {metrics['epoch']}] RMSE_log={metrics['RMSE_log']:.4f} "
              f"R2_log={metrics['R2_log']:.4f} parse_rate={metrics['parse_success_rate']*100:.1f}%")
        model.train()
        return control


def make_model_with_lora(params):
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=TRAIN_DTYPE)
    model.config.use_cache = False
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, inference_mode=False,
        r=params["r"], lora_alpha=params["lora_alpha"], lora_dropout=params["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    return get_peft_model(model, lora_config)


def tokenize_split(df):
    hf_ds = Dataset.from_pandas(df[["text", "target"]], preserve_index=False)
    return hf_ds.map(build_training_example, remove_columns=hf_ds.column_names)


def train_and_eval(params, train_df_local, eval_df_local, output_dir, run_label="", epoch_metrics_csv=None):
    cleanup_gpu()
    model = make_model_with_lora(params)
    tokenized_train = tokenize_split(train_df_local)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=params["num_train_epochs"],
        per_device_train_batch_size=params["train_batch_size"],
        gradient_accumulation_steps=params["gradient_accumulation_steps"],
        learning_rate=params["learning_rate"],
        warmup_ratio=params["warmup_ratio"],
        weight_decay=params["weight_decay"],
        lr_scheduler_type=params["lr_scheduler_type"],
        logging_steps=50,
        save_strategy="epoch",          # checkpoint every epoch -> resumable
        save_total_limit=2,             # keep disk usage bounded
        eval_strategy="no",
        bf16=USE_BF16, fp16=USE_FP16,
        optim="adamw_torch",
        report_to="none",
        remove_unused_columns=False,
        seed=SEED, data_seed=SEED,
        disable_tqdm=True,
    )

    callbacks = []
    if epoch_metrics_csv is not None:
        callbacks.append(EpochMetricsCallback(eval_df_local, epoch_metrics_csv))

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=tokenized_train, data_collator=data_collator,
        callbacks=callbacks,
    )

    # Resume automatically if a checkpoint already exists in output_dir (session-death recovery)
    resume_ckpt = None
    if os.path.isdir(output_dir):
        existing = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
        if existing:
            resume_ckpt = True
            print(f"Found existing checkpoint(s) in {output_dir} -> resuming training.")

    torch.cuda.empty_cache()
    t0 = time.perf_counter()
    trainer.train(resume_from_checkpoint=resume_ckpt)
    training_time = time.perf_counter() - t0

    pred_log, valid_mask, generated_texts, gen_time = generate_predictions(trainer.model, eval_df_local)
    y_true_log = eval_df_local["target"].to_numpy(dtype=float)
    actual_usd = eval_df_local["target_usd"].to_numpy(dtype=float)
    pred_usd = np.clip(np.expm1(pred_log), a_min=0, a_max=None)

    metrics = evaluate_regression_full(y_true_log, pred_log, actual_usd, pred_usd)
    metrics["Training_Time_Seconds"] = training_time
    metrics["Generation_Time_Seconds"] = gen_time
    metrics["Parse_Success_Rate"] = float(valid_mask.mean())

    print(f"[{run_label}] RMSE_log={metrics['RMSE_log']:.4f} MAE_log={metrics['MAE_log']:.4f} "
          f"R2_log={metrics['R2_log']:.4f} parse_rate={metrics['Parse_Success_Rate']*100:.1f}%")

    return metrics, trainer.model, pred_log, valid_mask, generated_texts


---
# Run baseline (fixed hyperparameters), full metrics per epoch + final


In [ ]:
baseline_epoch_csv = f"{OUTPUT_DIR}/baseline_epoch_metrics.csv"

baseline_metrics, baseline_model, baseline_pred_log, baseline_valid_mask, baseline_gen_texts = train_and_eval(
    BASELINE_PARAMS,
    train_df_local=train_df,
    eval_df_local=test_df,
    output_dir=CHECKPOINT_DIR,
    run_label="BASELINE/test",
    epoch_metrics_csv=baseline_epoch_csv,
)

baseline_result_row = {
    "Model": f"Qwen2-0.5B-LoRA-Baseline-{SERIALIZATION}",
    "Serialization": SERIALIZATION,
    "Stage": "baseline",
    **BASELINE_PARAMS,
    **baseline_metrics,
}
baseline_final_df = pd.DataFrame([baseline_result_row])
baseline_final_df.to_csv(f"{OUTPUT_DIR}/baseline_final_metrics.csv", index=False)
display(baseline_final_df)


## Save model (for XAI reuse later) + handoff config for the Optuna notebook

In [ ]:
# Model + tokenizer saved in HF/safetensors format -- directly usable by
# attention-visualization / gradient-attribution XAI tools later.
baseline_model.save_pretrained(BASELINE_MODEL_DIR)
tokenizer.save_pretrained(BASELINE_MODEL_DIR)
print("Saved baseline model to:", BASELINE_MODEL_DIR)

# Handoff file: lets the Optuna notebook rebuild the EXACT same train/val split
# and reuse the same fallback/clip values, so all metrics stay comparable.
shared_config = {
    "serialization": SERIALIZATION,
    "train_file": TRAIN_FILE,
    "test_file": TEST_FILE,
    "fallback_log_target": fallback_log_target,
    "max_reasonable_log_target": max_reasonable_log_target,
    "val_fraction": VAL_FRACTION,
    "seed": SEED,
    "train_ids": train_df["id"].tolist(),
    "val_ids": val_df["id"].tolist(),
}
with open(f"{OUTPUT_DIR}/shared_config.json", "w") as f:
    json.dump(shared_config, f)
print("Saved shared_config.json with", len(shared_config["train_ids"]), "train IDs and",
      len(shared_config["val_ids"]), "val IDs.")

baseline_pred_df = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "actual_log_target": test_df["target"].to_numpy(),
    "predicted_log_target": baseline_pred_log,
    "raw_generated_text": baseline_gen_texts,
    "numeric_parse_valid": baseline_valid_mask,
    "actual_usd": test_df["target_usd"].to_numpy(),
    "predicted_usd": np.clip(np.expm1(baseline_pred_log), 0, None),
})
baseline_pred_df.to_csv(f"{OUTPUT_DIR}/baseline_predictions.csv", index=False)
print("Saved baseline_predictions.csv")


---
# Next step

**Save this notebook's version** (Save Version → Save & Run All), then in the Optuna
notebook: **Add Input → Notebooks → this notebook** → its files will be available at
`/kaggle/input/<this-notebook-slug>/...` including `shared_config.json`, `baseline_model/`,
`baseline_epoch_metrics.csv`, `baseline_final_metrics.csv`.
